# 1. Exploração de Dados — Thermal Predictor

Estatistica descritiva das colunas: 

## 1. Estatística descritiva das colunas

A partir da classificação de colunas feita em [`classify_columns.ipynb`](classify_columns.ipynb)
(numéricas: `temp`, `active_limit`; categóricas: as demais 12 colunas), calculei aqui as
métricas descritivas de cada uma: média, desvio-padrão e quartis para as numéricas; valores
únicos, moda e frequência para as categóricas.

Fonte: `data/dataset_long.csv` (gerado por
[`00_reestruturacao_planilha.ipynb`](00_reestruturacao_planilha.ipynb)).

**Escopo desta seção:** apenas cálculo das métricas e montagem da tabela, nenhum dado é
tratado, filtrado ou interpretado aqui. Inconsistências já identificadas na classificação
(ex.: o valor residual de texto em `active_limit`, as grafias diferentes de
`blower_direction`) são reportadas como estão; o tratamento delas fica para a Seção 4.2.2
(pré-processamento).

In [1]:
import pandas as pd

CAMINHO = "../data/dataset_long.csv"

df = pd.read_csv(CAMINHO)
print(f"Dataset: {df.shape[0]} linhas, {df.shape[1]} colunas\n")

COLUNAS_NUMERICAS = ["temp", "active_limit"]
COLUNAS_CATEGORICAS = ["run_id", "tc_number", "tc_name", "key_area", "reason",
                        "blower_direction", "housing", "board_position",
                        "run_case", "losses", "deflector_hole",
                        "material_coil_support"]


# =============================================================================
# 1. ESTATÍSTICA DESCRITIVA - COLUNAS NUMÉRICAS
# =============================================================================
def estatisticas_numericas(dataframe, colunas):
    """
    Média, mediana, desvio-padrão, min, max, Q1 e Q3 de cada coluna numérica.
    `active_limit` é convertida com pd.to_numeric(errors='coerce'): o único
    valor não numérico da coluna ("Monitor (150 C)", 56 ocorrências, já
    documentado na classificação do Card 1.1) vira NaN só para permitir o
    cálculo — nenhum tratamento além disso é feito aqui.
    """
    df_numerico = dataframe[colunas].apply(pd.to_numeric, errors="coerce")
    return df_numerico.describe(percentiles=[.25, .5, .75]).round(2)


print("=" * 60)
print("1. ESTATISTICA DESCRITIVA - COLUNAS NUMERICAS")
print("=" * 60)
tab_num = estatisticas_numericas(df, COLUNAS_NUMERICAS)
print(tab_num.to_string())

# =============================================================================
# 2. ESTATÍSTICA DESCRITIVA - COLUNAS CATEGÓRICAS
# =============================================================================
def estatisticas_categoricas(dataframe, colunas):
    """
    Moda, número de entradas, valores nulos e valores únicos de cada
    coluna categórica. Retorna um resumo consolidado (uma linha por coluna).
    """
    linhas = []
    for col in colunas:
        s = dataframe[col]
        linhas.append({
            "coluna": col,
            "moda": s.mode()[0] if not s.mode().empty else None,
            "entradas": s.count(),
            "nulos": s.isna().sum(),
            "unicos": s.nunique(),
        })
    return pd.DataFrame(linhas)


print("\n" + "=" * 60)
print("2. ESTATISTICA DESCRITIVA - COLUNAS CATEGORICAS (resumo)")
print("=" * 60)
tab_cat = estatisticas_categoricas(df, COLUNAS_CATEGORICAS)
print(tab_cat.to_string(index=False))

# Frequência completa (value_counts) das categóricas de baixa/média
# cardinalidade — para run_id, tc_number e tc_name (56-77 valores únicos
# cada) o resumo acima já cobre o pedido; a distribuição completa deles
# fica extensa demais para caber em tabela e não agrega além do resumo.
COLUNAS_PARA_FREQUENCIA = ["key_area", "reason", "blower_direction", "housing",
                            "board_position", "run_case", "losses",
                            "deflector_hole", "material_coil_support"]

print("\n" + "=" * 60)
print("FREQUÊNCIA (value_counts) POR COLUNA")
print("=" * 60)
for col in COLUNAS_PARA_FREQUENCIA:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

Dataset: 4065 linhas, 14 colunas

1. ESTATISTICA DESCRITIVA - COLUNAS NUMERICAS
          temp  active_limit
count  4065.00       4009.00
mean     81.77        184.43
std      33.28        235.17
min      27.69         60.00
25%      59.11        100.00
50%      76.50        110.00
75%      96.02        150.00
max     273.60       1000.00

2. ESTATISTICA DESCRITIVA - COLUNAS CATEGORICAS (resumo)
               coluna                    moda  entradas  nulos  unicos
               run_id                  Run 20      4065      0      56
            tc_number                       1      4065      0      77
              tc_name AC Blower HES Connector      4065      0      77
             key_area                IPC Comp      4065      0       7
               reason       Baseline new Duct      3931    134      34
     blower_direction                To Front      4065      0       4
              housing                  Closed      4065      0       3
       board_position            